In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from astropy import units as u
from astropy import constants as const

In [2]:
solar_system = pd.read_csv('solar_system.csv')
print(solar_system.shape)

(20, 11)


In [3]:
solar_system = solar_system.set_index('Attribute').T #attributes become index labels instead of a normal column
#.T transposes the df so rows become columns and columns become rows
solar_system.index.name = 'Planet' #renames the index to 'Planet'
solar_system.reset_index(inplace=True) #converts the index back to a normal column
solar_system.columns.name = None #clears column axis names

print(solar_system.shape)

(10, 21)


In [4]:
print(solar_system)

    Planet Mass (10^24kg) Diameter (km) Density (kg/m^3) Gravity (m/s^2)  \
0  Mercury          0.330          4879             5429             3.7   
1    Venus           4.87         12104             5243             8.9   
2    Earth           5.97         12756             5514             9.8   
3     Moon          0.073          3475             3340             1.6   
4     Mars          0.642          6792             3934             3.7   
5  Jupiter           1898        142984             1326            23.1   
6   Saturn            568        120536              687             9.0   
7   Uranus           86.8         51118             1270             8.7   
8  Neptune            102         49528             1638            11.0   
9    Pluto         0.0130          2376             1850             0.7   

  Escape Velocity (km/s) Rotation Period (hours) Length of Day (hours)  \
0                    4.3                  1407.6                4222.6   
1              

In [5]:
print(solar_system.dtypes)

Planet                         object
Mass (10^24kg)                 object
Diameter (km)                  object
Density (kg/m^3)               object
Gravity (m/s^2)                object
Escape Velocity (km/s)         object
Rotation Period (hours)        object
Length of Day (hours)          object
Distance from Sun (10^6 km)    object
Perihelion (10^6 km)           object
Aphelion (10^6 km)             object
Orbital Period (days)          object
Orbital Velocity (km/s)        object
Orbital Inclination (deg)      object
Orbital Eccentricity           object
Obliquity to Orbit (deg)       object
Mean Temperature (C)           object
Surface Pressure (bars)        object
Number of Moons                object
Ring System?                   object
Global Magnetic Field?         object
dtype: object


In [6]:
for col in solar_system.columns:
    types = solar_system[col].apply(type)
    print(f"Column: {col}")
    print(types)

Column: Planet
0    <class 'str'>
1    <class 'str'>
2    <class 'str'>
3    <class 'str'>
4    <class 'str'>
5    <class 'str'>
6    <class 'str'>
7    <class 'str'>
8    <class 'str'>
9    <class 'str'>
Name: Planet, dtype: object
Column: Mass (10^24kg)
0    <class 'str'>
1    <class 'str'>
2    <class 'str'>
3    <class 'str'>
4    <class 'str'>
5    <class 'str'>
6    <class 'str'>
7    <class 'str'>
8    <class 'str'>
9    <class 'str'>
Name: Mass (10^24kg), dtype: object
Column: Diameter (km)
0    <class 'str'>
1    <class 'str'>
2    <class 'str'>
3    <class 'str'>
4    <class 'str'>
5    <class 'str'>
6    <class 'str'>
7    <class 'str'>
8    <class 'str'>
9    <class 'str'>
Name: Diameter (km), dtype: object
Column: Density (kg/m^3)
0    <class 'str'>
1    <class 'str'>
2    <class 'str'>
3    <class 'str'>
4    <class 'str'>
5    <class 'str'>
6    <class 'str'>
7    <class 'str'>
8    <class 'str'>
9    <class 'str'>
Name: Density (kg/m^3), dtype: object
Column: Gravity (m

The actual data types are strings, which is a problem for data analysis because you can't use the values directly with numpy or curve fitting or other types of stuff.

In [7]:
for col in solar_system.columns:
    solar_system[col] = pd.to_numeric(solar_system[col], errors='coerce')

print(solar_system.dtypes)

Planet                         float64
Mass (10^24kg)                 float64
Diameter (km)                    int64
Density (kg/m^3)                 int64
Gravity (m/s^2)                float64
Escape Velocity (km/s)         float64
Rotation Period (hours)        float64
Length of Day (hours)          float64
Distance from Sun (10^6 km)    float64
Perihelion (10^6 km)           float64
Aphelion (10^6 km)             float64
Orbital Period (days)          float64
Orbital Velocity (km/s)        float64
Orbital Inclination (deg)      float64
Orbital Eccentricity           float64
Obliquity to Orbit (deg)       float64
Mean Temperature (C)             int64
Surface Pressure (bars)        float64
Number of Moons                  int64
Ring System?                   float64
Global Magnetic Field?         float64
dtype: object


float64: a floating point decimal number

int64: an integer

column names are strings, so they are still objects

In [8]:
def attach_units(df, col):
    df[col] = df[col].values * 1e6 * u.km
    
attach_units(solar_system, "Distance from Sun (10^6 km)")

In [9]:
print(solar_system.columns)

Index(['Planet', 'Mass (10^24kg)', 'Diameter (km)', 'Density (kg/m^3)',
       'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)',
       'Length of Day (hours)', 'Distance from Sun (10^6 km)',
       'Perihelion (10^6 km)', 'Aphelion (10^6 km)', 'Orbital Period (days)',
       'Orbital Velocity (km/s)', 'Orbital Inclination (deg)',
       'Orbital Eccentricity', 'Obliquity to Orbit (deg)',
       'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons',
       'Ring System?', 'Global Magnetic Field?'],
      dtype='object')


In [10]:
print(solar_system["Distance from Sun (10^6 km)"].values)

[5.7900e+07 1.0820e+08 1.4960e+08 3.8400e+05 2.2800e+08 7.7850e+08
 1.4320e+09 2.8670e+09 4.5150e+09 5.9064e+09] km


In [11]:
#compute the average of perhelion and aphelion, and add this as a new column after aphelion using df.insert()
semi_major_axis = (solar_system["Perihelion (10^6 km)"] + solar_system["Aphelion (10^6 km)"]) / 2

aphelion_index = solar_system.columns.get_loc("Aphelion (10^6 km)")
insert_index = aphelion_index + 1

solar_system.insert(insert_index, "Semi-Major Axis (10^6 km)", semi_major_axis)

print(solar_system.columns)
print(solar_system["Semi-Major Axis (10^6 km)"].values)

Index(['Planet', 'Mass (10^24kg)', 'Diameter (km)', 'Density (kg/m^3)',
       'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)',
       'Length of Day (hours)', 'Distance from Sun (10^6 km)',
       'Perihelion (10^6 km)', 'Aphelion (10^6 km)',
       'Semi-Major Axis (10^6 km)', 'Orbital Period (days)',
       'Orbital Velocity (km/s)', 'Orbital Inclination (deg)',
       'Orbital Eccentricity', 'Obliquity to Orbit (deg)',
       'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons',
       'Ring System?', 'Global Magnetic Field?'],
      dtype='object')
[5.79000e+01 1.08200e+02 1.49600e+02 3.84500e-01 2.28000e+02 7.78500e+02
 1.43205e+03 2.86705e+03 4.51500e+03 5.90635e+03]


In [12]:
orbital_period_years = (solar_system["Orbital Period (days)"]) / 365
solar_system["Orbital Period (days)"] = orbital_period_years
solar_system.rename(columns={"Orbital Period (days)": "Orbital Period (years)"}, inplace=True)

print(solar_system.columns)
print(solar_system["Orbital Period (years)"].values)

Index(['Planet', 'Mass (10^24kg)', 'Diameter (km)', 'Density (kg/m^3)',
       'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)',
       'Length of Day (hours)', 'Distance from Sun (10^6 km)',
       'Perihelion (10^6 km)', 'Aphelion (10^6 km)',
       'Semi-Major Axis (10^6 km)', 'Orbital Period (years)',
       'Orbital Velocity (km/s)', 'Orbital Inclination (deg)',
       'Orbital Eccentricity', 'Obliquity to Orbit (deg)',
       'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons',
       'Ring System?', 'Global Magnetic Field?'],
      dtype='object')
[2.41095890e-01 6.15616438e-01 1.00054795e+00 7.47945205e-02
 1.88219178e+00 1.18657534e+01 2.94438356e+01 8.38054795e+01
 1.63835616e+02 2.48109589e+02]


In [13]:
value = solar_system.loc[6, "Orbital Period (years)"]
print(f"Saturn's orbital period: {value:.4f}")

Saturn's orbital period: 29.4438


In [14]:
#One astronomical unit's length is equal to the average distance between the Earth and Sun
print(const.au)
print((const.au).to(u.km))

  Name   = Astronomical Unit
  Value  = 149597870700.0
  Uncertainty  = 0.0
  Unit  = m
  Reference = IAU 2012 Resolution B2
149597870.70000002 km


In [15]:
print(solar_system.columns)

Index(['Planet', 'Mass (10^24kg)', 'Diameter (km)', 'Density (kg/m^3)',
       'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)',
       'Length of Day (hours)', 'Distance from Sun (10^6 km)',
       'Perihelion (10^6 km)', 'Aphelion (10^6 km)',
       'Semi-Major Axis (10^6 km)', 'Orbital Period (years)',
       'Orbital Velocity (km/s)', 'Orbital Inclination (deg)',
       'Orbital Eccentricity', 'Obliquity to Orbit (deg)',
       'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons',
       'Ring System?', 'Global Magnetic Field?'],
      dtype='object')


In [16]:
solar_system["Distance from Sun (10^6 km)"] = (solar_system["Distance from Sun (10^6 km)"].values).to(u.AU).value
solar_system.rename(columns={"Distance from Sun (10^6 km)": "Distance from Sun (AU)"}, inplace=True)

solar_system["Perihelion (10^6 km)"] = (solar_system["Perihelion (10^6 km)"].values * 1e6 * u.km).to(u.AU).value
solar_system.rename(columns={"Perihelion (10^6 km)": "Perihelion (AU)"}, inplace=True)

solar_system["Aphelion (10^6 km)"] = (solar_system["Aphelion (10^6 km)"].values * 1e6 * u.km).to(u.AU).value
solar_system.rename(columns={"Aphelion (10^6 km)": "Aphelion (AU)"}, inplace=True)

solar_system["Semi-Major Axis (10^6 km)"] = (solar_system["Semi-Major Axis (10^6 km)"].values * 1e6 * u.km).to(u.AU).value
solar_system.rename(columns={"Semi-Major Axis (10^6 km)": "Semi-Major Axis (AU)"}, inplace=True)

In [17]:
print(solar_system)

   Planet  Mass (10^24kg)  Diameter (km)  Density (kg/m^3)  Gravity (m/s^2)  \
0     NaN           0.330           4879              5429              3.7   
1     NaN           4.870          12104              5243              8.9   
2     NaN           5.970          12756              5514              9.8   
3     NaN           0.073           3475              3340              1.6   
4     NaN           0.642           6792              3934              3.7   
5     NaN        1898.000         142984              1326             23.1   
6     NaN         568.000         120536               687              9.0   
7     NaN          86.800          51118              1270              8.7   
8     NaN         102.000          49528              1638             11.0   
9     NaN           0.013           2376              1850              0.7   

   Escape Velocity (km/s)  Rotation Period (hours)  Length of Day (hours)  \
0                     4.3                   1407.6   

In [19]:
solar_system.to_csv("units.csv", index=False)